# EDA — Combined · TTC Routes 29 & 39 · Bus Delay Prediction
**DAMO-699 Capstone Project**  
Team: Saurav · Kriti · Pooja · Shristi &nbsp;|&nbsp; Supervisor: Dr. Bilal El Toufaili  
Deadline: March 20, 2026

---
This notebook runs **both routes side-by-side** for direct comparison.  
Use the per-route notebooks (`notebooks/29/02_eda.ipynb` and `notebooks/39/02_eda.ipynb`)  
for detailed single-route work. This notebook is for cross-route comparison only.


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 4)

DATA_DIR = Path('../data')
ROUTES   = [29, 39]
COLORS   = {29: '#1f77b4', 39: '#ff7f0e'}


## Load Data (Both Routes)

In [10]:
# Load your processed dataframes for each route:
dfs = {}
for r in ROUTES:
    dfs[r] = pd.read_parquet(DATA_DIR / f'processed/route{r}_stop_events.parquet')
    dfs[r]['route'] = r
df_all = pd.concat(dfs.values(), ignore_index=True)
print("Combined shape:", df_all.shape)
print(df_all.groupby('route').size())
print("TODO: load both route dataframes")


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/route29_stop_events.parquet'

## Time-Tagging (Both Routes)

In [ ]:
def tag_time(df, ts='timestamp'):
    df = df.copy()
    df['hour_of_day'] = df[ts].dt.hour
    df['day_of_week'] = df[ts].dt.dayofweek
    df['is_weekday']  = df['day_of_week'] < 5
    bins   = [-1, 4, 6, 8, 14, 18, 21, 23]
    labels = ['Late Night','Early Morning','AM Peak','Midday','PM Peak','Evening','Late Night2']
    df['time_bucket'] = (pd.cut(df['hour_of_day'], bins=bins, labels=labels)
                           .astype(str).replace('Late Night2','Late Night'))
    return df
# df_all = tag_time(df_all)
print("TODO: apply tag_time()")


## Comparison 1 — Delay Distribution by Route

In [ ]:
# ── Side-by-side histograms ─────────────────────────────────────────────────
# delay_col = 'delay_sec'
# fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# for ax, r in zip(axes, ROUTES):
#     df = dfs[r]
#     ax.hist(df[delay_col].dropna(), bins=100, color=COLORS[r], edgecolor='none', alpha=0.8)
#     ax.set_title(f'Route {r} — Delay Distribution')
#     ax.set_xlabel('Delay (s)')
#     stats = df[delay_col].describe(percentiles=[.5,.9])
#     ax.axvline(stats['mean'], color='red', ls='--', label=f"mean={stats['mean']:.0f}s")
#     ax.axvline(stats['50%'], color='green', ls='--', label=f"median={stats['50%']:.0f}s")
#     ax.legend(fontsize=8)
# plt.tight_layout(); plt.show()
# print(df_all.groupby('route')[delay_col].describe(percentiles=[.5,.9]))
print("TODO C1: delay distribution comparison")


## Comparison 2 — Mean Delay by Time of Day

In [ ]:
# ── Hour-of-day comparison ─────────────────────────────────────────────────
# hourly = df_all.groupby(['route','hour_of_day'])[delay_col].mean().reset_index()
# for r, g in hourly.groupby('route'):
#     plt.plot(g['hour_of_day'], g[delay_col], marker='o', color=COLORS[r], label=f'Route {r}')
# plt.xlabel('Hour of Day'); plt.ylabel('Mean Delay (s)')
# plt.title('Routes 29 vs 39 — Mean Delay by Hour')
# plt.legend(); plt.tight_layout(); plt.show()
print("TODO C2: hourly comparison")


## Comparison 3 — Bunching Rate by Route

In [ ]:
# ── Bunching comparison ────────────────────────────────────────────────────
# for r in ROUTES:
#     df = dfs[r]
#     if 'scheduled_headway_sec' in df.columns and 'actual_headway_sec' in df.columns:
#         bunching_rate = (df['actual_headway_sec'] < 0.5 * df['scheduled_headway_sec']).mean()
#         print(f"Route {r} bunching rate: {bunching_rate*100:.1f}%")
print("TODO C3: bunching comparison")


## Comparison 4 — Stop-Level Delay Profiles

In [ ]:
# ── Stop-sequence delay profiles (NB direction only for clarity) ──────────
# delay_col = 'delay_sec'
# for r in ROUTES:
#     nb = dfs[r][dfs[r]['direction'] == 'Northbound']
#     s  = nb.groupby('stop_sequence')[delay_col].mean()
#     plt.plot(s.index, s.values, color=COLORS[r], label=f'Route {r}', marker='.')
# plt.xlabel('Stop Sequence'); plt.ylabel('Mean Delay (s)')
# plt.title('Northbound Delay by Stop Sequence')
# plt.legend(); plt.tight_layout(); plt.show()
print("TODO C4: stop-sequence comparison")


## Comparison 5 — Cascading Effect Correlation

In [ ]:
# ── Previous-trip-delay correlation ────────────────────────────────────────
# for r in ROUTES:
#     df = dfs[r].copy()
#     vt = (df.sort_values(['vehicle_id','timestamp'])
#             .groupby(['vehicle_id','trip_id'])
#             .agg(trip_start=('timestamp','min'), end_delay=(delay_col,'last'))
#             .reset_index().sort_values(['vehicle_id','trip_start']))
#     vt['prev'] = vt.groupby('vehicle_id')['end_delay'].shift(1)
#     corr = vt[['end_delay','prev']].corr().iloc[0,1]
#     print(f"Route {r} — cascade corr end_delay[N] vs [N-1]: {corr:.3f}")
print("TODO C5: cascading correlation comparison")


## Summary — Cross-Route Comparison

In [ ]:
# ── Summary statistics table ────────────────────────────────────────────────
# summary_rows = []
# for r in ROUTES:
#     df = dfs[r]
#     summary_rows.append({
#         'route': r,
#         'mean_delay_s':  df[delay_col].mean(),
#         'median_delay_s': df[delay_col].median(),
#         'p90_delay_s':   df[delay_col].quantile(0.9),
#         'pct_early':     (df[delay_col] < 0).mean()*100,
#         'pct_gt10min':   (df[delay_col] > 600).mean()*100,
#     })
# summary = pd.DataFrame(summary_rows).set_index('route')
# print(summary.round(1))
print("TODO: fill in summary table")


## Key Questions — Cross-Route

| Question | Route 29 | Route 39 |
|---|---|---|
| Peak delay (AM or PM)? | | |
| Worst stop (stop_id & location)? | | |
| Bunching rate? | | |
| Cascade correlation? | | |
| Strongest feature (by \\|r\\|)? | | |

---
*Data Sources: TTC VISION GPS (AVL) · GTFS scheduled stop times · TransSee stop-level data*  
*Reference: Steve Munro — Methodology for Analysis of TTC Vehicle Tracking Data*
